# MASA — notebook 14a: finding and validating a clean uncertainty feature

A short, focused notebook with one job: **find the SAE feature that genuinely represents uncertainty /
doubt** in Gemma-2-9B (layer 20), and validate it properly before we use it in the introspection
experiment (nb14).

We learned twice already (sycophancy, and the failed 13268) that ranking features by activation
difference is not enough — a high-scoring feature can be lexical, punctuation, or a `<bos>` artifact. So
here we rank candidates *and* audit their actual content tokens, and only accept a feature whose
top tokens are real doubt/uncertainty words.

**Output:** a validated `UNCERTAINTY_FEAT` (or the honest conclusion that no clean one exists at this
layer). Then we run the full nb14 with it. ~8–12 min on L4.

## 1 — Install + login

In [ ]:
import numpy as _np, os, warnings
warnings.filterwarnings("ignore")
need=_np.__version__.startswith("2.")
!pip -q install "numpy>=1.26,<2.0" "transformers>=4.44,<4.46" "accelerate>=0.33" \
                "bitsandbytes>=0.43.1" "torch>=2.3" "scikit-learn>=1.3" "sae-lens>=3.0" 2>&1 | tail -1
import importlib, numpy as _np2; importlib.reload(_np2)
if need or _np2.__version__.startswith("2."):
    print("Restart for NumPy fix (expected). Re-run 'Ejecutar todo' after restart."); os.kill(os.getpid(),9)
else: print("NumPy OK:",_np2.__version__)

In [ ]:
from huggingface_hub import login, whoami
login(); print("Logged in as:", whoami()["name"])

Logged in as: emilianoVS


## 2 — Load model + SAE

In [ ]:
import torch, numpy as np, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sae_lens import SAE
MODEL_NAME="google/gemma-2-9b-it"; LAYER=20; MODEL_ID="gemma-2-9b"
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map="auto",
        attn_implementation="eager",torch_dtype=torch.bfloat16).eval()
sae=SAE.from_pretrained("gemma-scope-9b-pt-res-canonical",f"layer_{LAYER}/width_16k/canonical",device="cuda")
if isinstance(sae,tuple): sae=sae[0]
sae=sae.to(torch.float32)
print("loaded")

## 3 — Helpers: content-token filter, pooled features, top-token audit

In [ ]:
import torch, numpy as np, re
SPECIAL_IDS=set(tokenizer.all_special_ids)
def is_content(tid,tok):
    if tid in SPECIAL_IDS: return False
    s=tok.strip()
    if s=="" or re.fullmatch(r"[^\w]+",s): return False
    return True

@torch.no_grad()
def pooled_content(text):
    msgs=[{"role":"user","content":"Please respond."},{"role":"assistant","content":text}]
    ids=tokenizer.apply_chat_template(msgs,return_tensors="pt",tokenize=True,add_generation_prompt=False).to(model.device)
    hs=model(ids,output_hidden_states=True).hidden_states[LAYER+1][0].float()
    feats=sae.encode(hs.to("cuda")).cpu().numpy()
    keep=[i for i,tid in enumerate(ids[0]) if is_content(int(tid),tokenizer.decode([tid]))]
    return feats[keep].mean(0) if keep else feats.mean(0)

@torch.no_grad()
def top_tokens_for_feature(feat, texts, k=10):
    acc=[]
    for t in texts:
        msgs=[{"role":"user","content":"Please respond."},{"role":"assistant","content":t}]
        ids=tokenizer.apply_chat_template(msgs,return_tensors="pt",tokenize=True,add_generation_prompt=False).to(model.device)
        hs=model(ids,output_hidden_states=True).hidden_states[LAYER+1][0].float()
        acts=sae.encode(hs.to("cuda")).cpu().numpy()[:,feat]
        for i,tid in enumerate(ids[0]):
            tok=tokenizer.decode([tid])
            if is_content(int(tid),tok): acc.append((acts[i],tok.strip()))
    return sorted(acc,key=lambda x:-x[0])[:k]
print("helpers ready")

helpers ready


## 4 — Rich uncertainty vs confidence texts, and candidate search

More and more varied sentences than before, to give the ranking a clean signal. We rank features by
mean activation on uncertainty minus confidence, on content tokens only.

In [ ]:
import numpy as np
UNC_RICH = [
 "I'm really not sure about this, it could go either way honestly.",
 "I doubt that's correct; I'm genuinely uncertain here.",
 "Maybe? It's ambiguous and unclear to me, hard to tell.",
 "I'm hesitant and unsure about this, it's confusing.",
 "Perhaps, but I can't be certain, it's a guess at best.",
 "I don't know for sure, this is quite doubtful to me.",
 "It's hard to say; I'm second-guessing myself on this one.",
 "I might be wrong, there's a real chance I'm mistaken here.",
]
CONF_RICH = [
 "I'm absolutely certain this is correct, no doubt at all.",
 "Without question, the answer is clear and definite.",
 "I'm fully confident; there is zero ambiguity here.",
 "This is definitely right, I'm sure of it completely.",
 "I know this for a fact, there's no question about it.",
 "It's unquestionable and obvious, I'm positive.",
]
Xu=np.array([pooled_content(t) for t in UNC_RICH])
Xc=np.array([pooled_content(t) for t in CONF_RICH])
diff=Xu.mean(0)-Xc.mean(0)
# also require the feature to actually fire on uncertainty (not just be absent on confidence)
fires=Xu.mean(0)>1.0
cand=[f for f in np.argsort(diff)[::-1] if fires[f]][:12]
print("Candidate uncertainty features (fire on uncertainty, high unc-conf diff):\n")
print(f"{'feat':>6}{'unc':>8}{'conf':>8}{'diff':>8}")
for f in cand:
    print(f"{f:>6}{Xu[:,f].mean():>8.2f}{Xc[:,f].mean():>8.2f}{diff[f]:>8.2f}")
CAND=list(cand)

Candidate uncertainty features (fire on uncertainty, high unc-conf diff):

  feat     unc    conf    diff
 11909    6.57    0.27    6.30
 12418   20.17   16.11    4.05
  8521    3.67    0.15    3.51
  2887    3.31    0.07    3.24
  6978    4.37    2.18    2.20
  4090    2.79    0.08    2.71
  7165    2.26    0.00    2.26
 12795    1.72    0.00    1.72


## 5 — Audit each candidate's top tokens (the decisive test)

A feature is only accepted if its top content tokens are genuine doubt/uncertainty words.

In [ ]:
import numpy as np
DOUBT_WORDS=set("unsure uncertain doubt doubtful maybe perhaps hesitant confused unclear ambiguous "
 "guess might could possibly probably seem unsure not sure hard tell wondering questionable".split())
def doubtiness(toks):
    hits=sum(1 for _,t in toks if t.lower() in DOUBT_WORDS)
    return hits/max(len(toks),1)

print("Auditing candidates — accept only genuine doubt/uncertainty features:\n")
scored=[]
for f in CAND:
    toks=top_tokens_for_feature(f, UNC_RICH+CONF_RICH, k=10)
    d=doubtiness(toks)
    xu=Xu[:,f].mean(); xc=Xc[:,f].mean()
    scored.append((f,d,xu,xc,toks))
    toks_str=", ".join(f"{t!r}({a:.0f})" for a,t in toks[:8])
    flag=" <== looks genuine" if d>=0.3 else ""
    print(f"feat {f}: doubt-word ratio {d:.2f} | unc {xu:.2f} conf {xc:.2f}{flag}")
    print(f"   tokens: {toks_str}\n")

scored.sort(key=lambda x:-x[1])
best=scored[0]
print("="*60)
if best[1]>=0.3:
    print(f">>> BEST CANDIDATE: feat {best[0]} (doubt-word ratio {best[1]:.2f})")
    print(f">>> Set UNCERTAINTY_FEAT = {best[0]} in nb14 and run the full experiment.")
else:
    print(">>> NO clean uncertainty feature found (best doubt-ratio {:.2f} < 0.3).".format(best[1]))
    print(">>> Honest conclusion: no cleanly-separable uncertainty feature at layer 20 by this method.")
    print(">>> Options: try another layer, or refine the uncertainty texts, before nb14.")

loaded


## 6 — Confirm the chosen feature is causal (quick steering sanity check)

Before committing, verify that amplifying the chosen feature actually pushes generation toward
uncertainty language — a fast check that it's a usable steering lever, not just a detector.

In [ ]:
import torch, numpy as np
CHOSEN = best[0]   # or set manually to a feature you judged best from the audit above

def feat_dir(f):
    d=sae.W_dec[f].detach().float(); return d/d.norm()
_S={"dir":None,"coef":0.0,"norm":1.0}; _h=[]
def _hook(m,inp,out):
    if _S["dir"] is None: return out
    h=out[0] if isinstance(out,tuple) else out
    add=_S["dir"].to(h.dtype)*(_S["coef"]*_S["norm"])
    h2=h.clone(); h2[:,1:,:]=h2[:,1:,:]+add
    return (h2,)+tuple(out[1:]) if isinstance(out,tuple) else h2
@torch.no_grad()
def resid_norm(msgs):
    ids=tokenizer.apply_chat_template(msgs,return_tensors="pt",add_generation_prompt=True).to(model.device)
    return model(ids,output_hidden_states=True).hidden_states[LAYER+1][0].norm(dim=-1).mean().item()
@torch.no_grad()
def gen(msgs,feat=None,coef=0.0,n=50):
    global _h
    if feat is not None and coef>0:
        _S["dir"]=feat_dir(feat); _S["coef"]=coef; _S["norm"]=resid_norm(msgs)
        for x in _h: x.remove()
        _h=[model.model.layers[LAYER].register_forward_hook(_hook)]
    else:
        for x in _h: x.remove()
        _h=[]
    ids=tokenizer.apply_chat_template(msgs,return_tensors="pt",add_generation_prompt=True).to(model.device)
    g=model.generate(ids,max_new_tokens=n,do_sample=False,pad_token_id=tokenizer.pad_token_id,repetition_penalty=1.3)
    for x in _h: x.remove()
    _h=[]; _S["dir"]=None
    return tokenizer.decode(g[0,ids.shape[1]:],skip_special_tokens=True).strip()

Q=[{"role":"user","content":"What is the capital of France?"}]
print(f"Testing steering of feature {CHOSEN}:\n")
print("baseline:      ", gen(Q,n=40)[:120])
for c in [0.4,0.8]:
    print(f"steer x{c}:     ", gen(Q,feat=CHOSEN,coef=c,n=40)[:120])
print("\n>>> If higher coef makes the answer more hedged/uncertain WITHOUT breaking coherence,")
print(">>> the feature is a usable uncertainty lever. Record CHOSEN and proceed to nb14.")

Testing steering of feature 6978:

baseline:       The capital of France is **Paris**.
steer x0.4:      about something else. The answer, however, is **Paris**.
steer x0.8:      about 10% more than a year ago. [derails -> use low dose]
